In [1]:
import ollama

In [2]:
inventory_db = {
        "laptop": {"stock": 5, "base_price": 1200},
        "monitor": {"stock": 0, "base_price": 300},
        "keyboard": {"stock": 25, "base_price": 80}
    }

In [3]:
#DFINING TOOL 
# TOOL 1:CHECKING DATABASE
def check_inventory(product_name):
    product_name = product_name.lower()
    
    if product_name in inventory_db:
        return inventory_db[product_name]
    
    return {"stock": 0, "base_price": None}

# TOOL 2: Business Logic for Discounts
def calculate_loyalty_discount(base_price, years_as_customer):
    discount = min(years_as_customer * 0.05, 0.30)  # max 30%
    final_price = base_price * (1 - discount)
    return round(final_price, 2)

In [4]:
# 1. Map the functions so the script can call them by name
available_functions = {
    'check_inventory': check_inventory,
    'calculate_loyalty_discount': calculate_loyalty_discount,
}

In [5]:
#DEFINING SCHEMA
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_inventory",
            "description": "Get stock and price for a product",
            "parameters": {
                "type": "object",
                "properties": {
                    "product_name": {"type": "string"}
                },
                "required": ["product_name"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_loyalty_discount",
            "description": "Calculate final price based on loyalty years",
            "parameters": {
                "type": "object",
                "properties": {
                    "base_price": {"type": "number"},
                    "years_as_customer": {"type": "integer"}
                },
                "required": ["base_price", "years_as_customer"]
            }
        }
    }
]

In [6]:
tools

[{'type': 'function',
  'function': {'name': 'check_inventory',
   'description': 'Get stock and price for a product',
   'parameters': {'type': 'object',
    'properties': {'product_name': {'type': 'string'}},
    'required': ['product_name']}}},
 {'type': 'function',
  'function': {'name': 'calculate_loyalty_discount',
   'description': 'Calculate final price based on loyalty years',
   'parameters': {'type': 'object',
    'properties': {'base_price': {'type': 'number'},
     'years_as_customer': {'type': 'integer'}},
    'required': ['base_price', 'years_as_customer']}}}]

In [7]:
#“I am a customer for 5 years. What will be the final price of a laptop?"
#"I want to buy an iPhone. Can you check stock?"
message=[
        {"role": "user", "content":"I am a customer for 5 years. What will be the final price of a laptop?"}
    ]

#CALLING llM
response = ollama.chat(
    model="qwen3:8b",
    messages=message
   ,
    tools=tools
)

print(response)

model='qwen3:8b' created_at='2026-06-30T14:53:30.421002Z' done=True done_reason='stop' total_duration=83887698600 load_duration=11971743700 prompt_eval_count=207 prompt_eval_duration=4821014000 eval_count=390 eval_duration=67078278000 message=Message(role='assistant', content='', thinking='Okay, let\'s see. The user has been a customer for 5 years and wants to know the final price of a laptop. They mentioned their loyalty years, so I should check if there\'s a discount based on that.\n\nFirst, I need to use the calculate_loyalty_discount function. But wait, the function requires the base price and the years as a customer. The user provided the years (5), but not the base price. How do I get the base price of the laptop?\n\nOh, right, there\'s another function called check_inventory that gets the stock and price for a product. So I should first call check_inventory with the product name "laptop" to find out the base price. Once I have that, I can then use calculate_loyalty_discount with

In [8]:
response["message"]

Message(role='assistant', content='', thinking='Okay, let\'s see. The user has been a customer for 5 years and wants to know the final price of a laptop. They mentioned their loyalty years, so I should check if there\'s a discount based on that.\n\nFirst, I need to use the calculate_loyalty_discount function. But wait, the function requires the base price and the years as a customer. The user provided the years (5), but not the base price. How do I get the base price of the laptop?\n\nOh, right, there\'s another function called check_inventory that gets the stock and price for a product. So I should first call check_inventory with the product name "laptop" to find out the base price. Once I have that, I can then use calculate_loyalty_discount with the base price and 5 years to get the final price.\n\nWait, but the user didn\'t specify the laptop model. Maybe the function assumes a default or the most common laptop? Or maybe I need to ask for more details. But since the user hasn\'t pro

In [9]:
# Check if the model asked to use any tools
tool_calls = response['message'].get('tool_calls')

if tool_calls:
    for tool_call in tool_calls:
        tool_name = tool_call['function']['name']
        tool_args = tool_call['function']['arguments']

        # Get the actual Python function
        function_to_call = available_functions[tool_name]

        # Run the function with given arguments
        result = function_to_call(**tool_args)

        # Add the model's tool request to the conversation
        message.append(response['message'])

        # Add the tool's result back to the conversation
        message.append({
            'role': 'tool',
            'content': str(result)
        })

message

[{'role': 'user',
  'content': 'I am a customer for 5 years. What will be the final price of a laptop?'},
 Message(role='assistant', content='', thinking='Okay, let\'s see. The user has been a customer for 5 years and wants to know the final price of a laptop. They mentioned their loyalty years, so I should check if there\'s a discount based on that.\n\nFirst, I need to use the calculate_loyalty_discount function. But wait, the function requires the base price and the years as a customer. The user provided the years (5), but not the base price. How do I get the base price of the laptop?\n\nOh, right, there\'s another function called check_inventory that gets the stock and price for a product. So I should first call check_inventory with the product name "laptop" to find out the base price. Once I have that, I can then use calculate_loyalty_discount with the base price and 5 years to get the final price.\n\nWait, but the user didn\'t specify the laptop model. Maybe the function assumes a

In [10]:
# Ask the model to generate the final answer
final_response = ollama.chat(
    model='qwen3:8b',
    messages=message
)

print(final_response['message']['content'])

<tool_call>
{"name": "calculate_loyalty_discount", "arguments": {"base_price": 1200, "years_as_customer": 5}}
</tool_call>
